# FedOPF: FedAvg approach based on APPFL

* Fixed or add things in APPFL package
    - Add ```_load_data_opf()``` in ```appfl.agent.client.py```
    - Add ```load_parameters()``` for FedOPF in ```appfl.algorithm.trainer.base_trainer.py```
    - Add ```ACOPFTrainer()``` in ```appfl.algorithm.trainer.acopf_trainer.py``` and ```appfl.algorithm.trainer.__init__.py```

* For printing global model's parameter, we should modify the config files as below:
    - Set ```optimize_memory: False``` at ```server_configs```, ```scheduler_kwargs```, ```aggregator_kwargs```

In [1]:
import argparse
from omegaconf import OmegaConf
from appfl.agent import ClientAgent, ServerAgent

from torch_geometric.loader import DataLoader
from pypower.api import makeYbus
import torch
import time
import numpy as np

from collections import OrderedDict

[W102 08:24:28.077363402 Context.cpp:281] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


In [2]:
server_config_path = "./resources/configs/fedopf_server_fedavg.yaml"
client_config_path = "./resources/configs/fedopf_client_1.yaml"

# client_ids = [14, 57, 60, 73, 89, 118, 162, 197, 250, 793]
client_ids = [14, 57, 60, 73, 89, 118, 162, 197, 250, 793, 1354, 1664]
num_clients = len(client_ids)

In [3]:
# Load server agent config and set the # of clients
server_agent_config = OmegaConf.load(server_config_path)
server_agent_config.server_configs.num_clients = num_clients

In [4]:
server_agent_config

{'client_configs': {'train_configs': {'trainer': 'ACOPFTrainer', 'mode': 'epoch', 'num_local_epochs': 5, 'optim': 'Adam', 'optim_args': {'lr': 0.001, 'weight_decay': 0}, 'warmup_iter': 50, 'rho_init': 0.01, 'p_iter_max': 5, 'loss_fn_path': './resources/loss/ldloss.py', 'loss_fn_name': 'LDLoss', 'do_validation': False, 'do_pre_validation': False, 'metric_path': './resources/metric/acopf.py', 'metric_name': 'acopf_feasibility', 'use_dp': False, 'epsilon': 1, 'clip_grad': True, 'clip_value': 1.0, 'clip_norm': 2, 'train_batch_size': 50, 'val_batch_size': 1, 'train_data_shuffle': True, 'val_data_shuffle': False}, 'model_configs': {'model_path': './resources/model/graphopf.py', 'model_name': 'EAGNN', 'model_kwargs': {'n_gnn_layers': 3, 'hidden_dim': 40, 'K': 10, 'dropout': 0.1, 'concat': False, 'beta': True}}, 'comm_configs': {'compressor_configs': {'enable_compression': False, 'lossy_compressor': 'SZ2Compressor', 'lossless_compressor': 'blosc', 'error_bounding_mode': 'REL', 'error_bound': 0

In [5]:
# Create server agent
server_agent = ServerAgent(server_agent_config=server_agent_config)

appfl: ✅[2026-01-02 08:24:29,197 server]: Logging to ./output/result_Server_2026-01-02-08-24-29.txt


In [6]:
# Load base client configurations and set corresponding fields for different clients
client_agent_configs = [
    OmegaConf.load(client_config_path) for _ in range(num_clients)
]
for i, id in enumerate(client_ids):
    client_agent_configs[i].client_id = f"Client{i+1}"
    # client_agent_configs[i].data_configs.dataset_kwargs.num_clients = num_clients
    client_agent_configs[i].data_configs.dataset_kwargs.client_id = id
    # client_agent_configs[i].data_configs.dataset_kwargs.visualization = (
    #     True if i == 0 else False
    # )
    
    # only enable wandb for the first client is sufficient for logging all clients in serial run
    if hasattr(client_agent_configs[i], "wandb_configs") and client_agent_configs[i].wandb_configs.get("enable_wandb", False):
        if i == 0:
            client_agent_configs[i].wandb_configs.enable_wandb = True
        else:
            client_agent_configs[i].wandb_configs.enable_wandb = False

In [7]:
# Load client agents
client_agents = [
    ClientAgent(client_agent_config=client_agent_configs[i]) for i in range(num_clients)
]

appfl: ✅[2026-01-02 08:24:29,302 Client1]: Logging to ./output/result_Client1_2026-01-02-08-24-29.txt
/home/super/data1/skj/FedOPF-APPFL/FedAVG/resources/dataset/acopf_prob.py:71: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1728929558238/work/torch/csrc/utils/tensor_new.cpp:278.)
  self.slackva = torch.tensor([np.deg2rad(ppc['bus'][self.slack, idx_bus.VA])],
appfl: ✅[2026-01-02 08:24:29,524 Client2]: Logging to ./output/result_Client2_2026-01-02-08-24-29.txt
appfl: ✅[2026-01-02 08:24:30,043 Client3]: Logging to ./output/result_Client3_2026-01-02-08-24-30.txt
appfl: ✅[2026-01-02 08:24:30,447 Client4]: Logging to ./output/result_Client4_2026-01-02-08-24-30.txt
appfl: ✅[2026-01-02 08:24:30,928 Client5]: Logging to ./output/result_Client5_2026-01-02-08-24-30.txt
appfl: ✅[2026-01-02 08:24:31,

In [8]:
# Get additional client configurations from the server
client_config_from_server = server_agent.get_client_configs()
# print(OmegaConf.to_yaml(server_agent.get_client_configs()))
client_config_from_server.model_configs.model_name = 'GraphOPF'
client_config_from_server.model_configs.model_path = "./resources/model/graphopf_client.py"

for client_agent in client_agents:
    client_config_from_server.model_configs.model_kwargs.client_id = client_agent.dataset.nbus
    client_agent.load_config(client_config_from_server)

In [9]:
# Load initial global model from the server
init_global_model = server_agent.get_parameters(serial_run=True)
for client_agent in client_agents:
    client_agent.load_parameters(init_global_model)

In [10]:
# Main code: run FedAvg
round = 0
while not server_agent.training_finished():
    # Model save
    if round == server_agent_config.server_configs.num_global_epochs-1:
        for client_agent in client_agents:
            client_agent.save_checkpoint()
        server_agent.save_checkpoint()

    if round != 0:
        # Load the new global model from the server
        for client_agent, new_global_model_future in zip(client_agents, new_global_models):
            client_agent.load_parameters(new_global_model_future.result())

    new_global_models = []
    for client_agent in client_agents:
        # client local training
        client_agent.train()
        local_model = client_agent.get_parameters()
    
        if isinstance(local_model, tuple):
            local_model, metadata = local_model[0], local_model[1]
        else:
            metadata = {}
        ## NOTE: kj
        local_model = OrderedDict((k,v) for k,v in local_model.items() if k.startswith("layers"))
        # "Send" local model to server and get a Future object for the new global model
        # The Future object will be resolved when the server receives local models from all clients
        new_global_model_future = server_agent.global_update(
            client_id=client_agent.get_id(),
            local_model=local_model,
            blocking=False,
            **metadata,
        )
        new_global_models.append(new_global_model_future)
            
    round += 1
    # print(server_agent.get_parameters(init_model=False)['0.edge_aggr.0.weight'][:2,:])
    print(server_agent.get_parameters(init_model=False)['layers.0.edge_aggr.0.weight'][:2,:])

appfl: ✅[2026-01-02 08:24:53,972 Client1]:      Round      Epoch       Time Train Loss Train Accuracy
appfl: ✅[2026-01-02 08:24:54,712 Client1]:          0          0     0.7364     1.4128           40.0
appfl: ✅[2026-01-02 08:24:54,801 Client1]:          0          1     0.0874     1.3679           40.0
appfl: ✅[2026-01-02 08:24:54,883 Client1]:          0          2     0.0794     1.3223           40.0
appfl: ✅[2026-01-02 08:24:54,981 Client1]:          0          3     0.0967     1.2721           40.0
appfl: ✅[2026-01-02 08:24:55,073 Client1]:          0          4     0.0907     1.2111           40.0
appfl: ✅[2026-01-02 08:24:56,723 Client2]:      Round      Epoch       Time Train Loss Train Accuracy
appfl: ✅[2026-01-02 08:24:56,817 Client2]:          0          0     0.0902     6.7652       44.57143
appfl: ✅[2026-01-02 08:24:56,908 Client2]:          0          1     0.0896     6.5817       46.57143
appfl: ✅[2026-01-02 08:24:57,007 Client2]:          0          2     0.0976     6.

tensor([[ 0.2704,  0.2936, -0.0829,  0.3247, -0.0776,  0.0712, -0.1720,  0.2077],
        [ 0.3118, -0.2593,  0.3074,  0.0663,  0.2613,  0.0480,  0.1704, -0.0500]])


appfl: ✅[2026-01-02 08:26:17,918 Client1]:          1          0     0.0748     1.2508           40.0
appfl: ✅[2026-01-02 08:26:18,004 Client1]:          1          1     0.0837     1.1899           40.0
appfl: ✅[2026-01-02 08:26:18,099 Client1]:          1          2     0.0937     1.1167           40.0
appfl: ✅[2026-01-02 08:26:18,179 Client1]:          1          3     0.0793     1.0348           40.0
appfl: ✅[2026-01-02 08:26:18,269 Client1]:          1          4     0.0884     0.9366           40.0
appfl: ✅[2026-01-02 08:26:19,971 Client2]:          1          0     0.0802     5.9958      51.714287
appfl: ✅[2026-01-02 08:26:20,060 Client2]:          1          1     0.0868     5.7176      55.428574
appfl: ✅[2026-01-02 08:26:20,157 Client2]:          1          2     0.0951     5.4008       59.14286
appfl: ✅[2026-01-02 08:26:20,249 Client2]:          1          3     0.0899     5.0734       64.57143
appfl: ✅[2026-01-02 08:26:20,348 Client2]:          1          4     0.0972     4.

tensor([[ 0.2705,  0.2937, -0.0830,  0.3246, -0.0777,  0.0711, -0.1719,  0.2079],
        [ 0.3119, -0.2591,  0.3075,  0.0664,  0.2614,  0.0481,  0.1705, -0.0502]])


appfl: ✅[2026-01-02 08:27:39,438 Client1]:          2          0     0.0656     1.0833           40.0
appfl: ✅[2026-01-02 08:27:39,524 Client1]:          2          1     0.0847     0.9997           40.0
appfl: ✅[2026-01-02 08:27:39,616 Client1]:          2          2     0.0903     0.9044           40.0
appfl: ✅[2026-01-02 08:27:39,699 Client1]:          2          3     0.0810     0.7967           42.0
appfl: ✅[2026-01-02 08:27:39,786 Client1]:          2          4     0.0870     0.6885           54.8
appfl: ✅[2026-01-02 08:27:41,475 Client2]:          2          0     0.0912     5.3093      56.571426
appfl: ✅[2026-01-02 08:27:41,551 Client2]:          2          1     0.0746     4.9897      64.571434
appfl: ✅[2026-01-02 08:27:41,643 Client2]:          2          2     0.0902     4.6960       72.85715
appfl: ✅[2026-01-02 08:27:41,732 Client2]:          2          3     0.0870     4.4273       82.28572
appfl: ✅[2026-01-02 08:27:41,829 Client2]:          2          4     0.0954     4.

...

warm up end!


appfl: ✅[2026-01-02 13:11:25,887 Client11]:        199          0     3.0369   258.3449      64.100006
appfl: ✅[2026-01-02 13:11:28,878 Client11]:        199          1     2.9899   428.5033      51.323082
appfl: ✅[2026-01-02 13:11:31,890 Client11]:        199          2     3.0100   255.3272      57.507694
appfl: ✅[2026-01-02 13:11:34,872 Client11]:        199          3     2.9815   199.3002      68.653854
appfl: ✅[2026-01-02 13:11:37,880 Client11]:        199          4     3.0059   200.0966      63.653847


warm up end!


appfl: ✅[2026-01-02 13:11:44,608 Client12]:        199          0     4.6532    22.4247      98.846146
appfl: ✅[2026-01-02 13:11:49,006 Client12]:        199          1     4.3963    22.3825       99.12821
appfl: ✅[2026-01-02 13:11:53,398 Client12]:        199          2     4.3908    22.4028       98.58974
appfl: ✅[2026-01-02 13:11:57,777 Client12]:        199          3     4.3774    22.3757        99.5641
appfl: ✅[2026-01-02 13:12:02,184 Client12]:        199          4     4.4047    22.3641       99.46154


tensor([[ 0.2710,  0.2983, -0.0834,  0.3285, -0.0805,  0.0667, -0.1852,  0.2083],
        [ 0.3234, -0.2451,  0.3175,  0.0639,  0.2663,  0.0572,  0.1774, -0.0439]])


## Evaluate model on "test dataset" for each client

In [11]:
# Modifies stats in place
def dict_agg(stats, key, value, op='concat'):
    if key in stats.keys():
        if op == 'sum':
            stats[key] += value
        elif op == 'concat':
            stats[key] = np.concatenate((stats[key], value), axis=0)
        else:
            raise NotImplementedError
    else:
        stats[key] = value

In [12]:
for client_agent in client_agents:
    print(client_agent.get_id())
    test_dataloader = DataLoader(
            client_agent.test_dataset,
            batch_size=1,
            shuffle=False,
            drop_last=True
        )

    Ybus, Yf, Yt = makeYbus(client_agent.dataset.baseMVA, client_agent.dataset.ppc['bus'], client_agent.dataset.ppc['branch'])
    # branch thermal limit information
    flow_max = (client_agent.dataset.ppc['branch'][:, 5] / client_agent.dataset.baseMVA)**2
    flow_max[flow_max == 0] = np.inf # np.Inf
    flow_max = torch.tensor(flow_max, dtype=torch.float32).to(client_agent.dataset.device)

    test_len = 200
    node_means, node_stds, edge_means, edge_stds = client_agent.dataset.input_standardization(test_len, train=False) # (1, 2*nbus) <= for data normalization
    n_means = node_means.to(client_agent.dataset.device)
    n_stds = node_stds.to(client_agent.dataset.device)
    e_means = edge_means.to(client_agent.dataset.device)
    e_stds = edge_stds.to(client_agent.dataset.device)

    client_agent.model.eval()
    test_stats = {}
    test_eps_converge = 1e-4

    # LagM = torch.ones(1, 2*ng + 2*nbus + 2*nl).to(DEVICE) # shape: (1, num_inequalities)
    LagM_sp_g = torch.ones(1, 2).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_q_g = torch.ones(1, 2*client_agent.dataset.ng).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_v_m = torch.ones(1, 2*client_agent.dataset.nbus).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_line_l = torch.ones(1, 2*client_agent.dataset.nl).to(client_agent.dataset.device) # shape: (1, num_inequalities)

    solve_time = []
    with torch.no_grad():
        for (i, Xtest) in enumerate(test_dataloader):
            Xtest = Xtest.to(client_agent.dataset.device)

            start_time = time.time()
            Y = client_agent.model(Xtest, n_means, n_stds, e_means, e_stds)
            end_time = time.time()

            solve_time += [end_time - start_time]

            ## line thermal limit
            pg, qg, vm, va = client_agent.dataset.get_yvars(Y)
            vr = vm*torch.cos(va)
            vi = vm*torch.sin(va)
            vz = torch.complex(vr, vi) # complex voltage

            # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
            If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(client_agent.dataset.device) @ vz.T
            It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(client_agent.dataset.device) @ vz.T

            # Calculate the apparent power S
            Sf = vz[:,client_agent.dataset.ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
            St = vz[:,client_agent.dataset.ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
            Sff = Sf * torch.conj(Sf)
            Stt = St * torch.conj(St)

            # calculate the line thermal limit constraints violation
            diff_Sf = Sff.real - flow_max
            diff_St = Stt.real - flow_max
            # diff_Sf[torch.clamp(diff_Sf, 0) != 0]

            line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
            line_limit_vio_St = torch.clamp(diff_St, 0)
            ###########################################

            # test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM)
            test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = client_agent.loss_fn.forward(client_agent.dataset, Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)

            dict_agg(test_stats, 'time', np.array(end_time - start_time).reshape(1,-1))

            test_ineq_p_g = torch.cat([pg - client_agent.dataset.pmax.to(client_agent.dataset.device), client_agent.dataset.pmin.to(client_agent.dataset.device) - pg], dim=1)
            test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(client_agent.dataset.device)
            test_ineq_q_g = test_ineq_dist[:,2:2+2*client_agent.dataset.ng]
            test_ineq_v_m = test_ineq_dist[:,2+2*client_agent.dataset.ng:2+2*client_agent.dataset.ng+2*client_agent.dataset.nbus]
            test_ineq_line_l = test_ineq_dist[:,2+2*client_agent.dataset.ng+2*client_agent.dataset.nbus:]

            dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
            # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())

            dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())

            dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())

            pg_rate_torch = (((pg <= client_agent.dataset.pmax.to(client_agent.dataset.device)) & (pg >= client_agent.dataset.pmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.ng)*100
            qg_rate_torch = (((qg <= client_agent.dataset.qmax.to(client_agent.dataset.device)) & (qg >= client_agent.dataset.qmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.ng)*100
            dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
            # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())

            v_rate_torch = (((vm <= client_agent.dataset.vmax.to(client_agent.dataset.device)) & (vm >= client_agent.dataset.vmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.nbus)*100
            dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
            # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())

            sff_rate_torch = ((Sff.real <= flow_max).sum()/client_agent.dataset.nl)*100        
            stt_rate_torch = ((Stt.real <= flow_max).sum()/client_agent.dataset.nl)*100        
            dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())

            dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())

            test_eq_real = test_eq_resid[:,:client_agent.dataset.nbus]
            test_eq_react = test_eq_resid[:,client_agent.dataset.nbus:]
            dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:client_agent.dataset.nbus] <= 1e-2) & (test_eq_resid[:,:client_agent.dataset.nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:client_agent.dataset.nbus].shape[1]*100).detach().cpu().numpy())
            dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,client_agent.dataset.nbus:] <= 1e-2) & (test_eq_resid[:,client_agent.dataset.nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,client_agent.dataset.nbus:].shape[1]*100).detach().cpu().numpy())

    print("GraphLDE obj. value for test samples: ", np.round(np.mean(test_stats['test_obj_cost'])*10000, 4))
    print("GraphLDE eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
    print("GraphLDE eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
    print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
    print("GraphLDE eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
    print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
    print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))

    print("\n")
    print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
    print("GraphLDE ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
    print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
    print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
    print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
    print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
    print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
    print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
    print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
    print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))

    print("\n")
    print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
    print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
    print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
    print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
    print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
    print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
    print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

    print("\n")
    print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(test_stats['time']*1e3)))
    print('\n')

# [14, 57, 60, 73, 89, 118, 162, 197, 250, 793, 1354, 1664]

Client1
GraphLDE obj. value for test samples:  2184.3006
GraphLDE eq. mean for test samples:  9.626129e-06
GraphLDE eq. max for test samples:  9.188763e-05
GraphLDE eq. active mean for test samples:  8.195238e-06
GraphLDE eq. active max for test samples:  4.9397582e-05
GraphLDE eq. reactive mean for test samples:  1.105702e-05
GraphLDE eq. reactive max for test samples:  9.185718e-05


GraphLDE ineq. mean for test samples:  1.4695218e-05
GraphLDE ineq. max for test samples:  0.0011235306
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  0.00011756174
GraphLDE ineq. q_g max for test samples:  0.0011235306
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0
GraphLDE ineq. line_l max for test samples:  0.0


GraphLDE p_g satisfication rate for test samples:  100.0
GraphLDE q_g satisfication rate for test samples:  